## Global statistics on travel time to forest

In [ ]:
import sys
sys.path.insert(0, '../code')
import paths  # input and output locations, from config.yaml
import rasterio
import numpy as np
import pandas as pd
import scipy.stats as stats
import sys
sys.path.insert(0, '../code')
from constants import CONTINENT_COLORS, REGION_TO_CONTINENT

### 1. Population-weighted mean and distribution

In [ ]:
# Load NATIVE-resolution rasters (~1km), aligned to their overlap.
# NOTE: previously this cell used the pre-aggregated 50km-block rasters
# for tractability. That was found to materially bias both this mean and
# the Gini coefficient below (see cell 2): the 50km version reported
# ~30.2 min / Gini 0.60 vs. ~21.7 min / Gini 0.70 at native resolution.
# Loading native full-resolution arrays (~67M valid pixels) is slower
# (~1-2 min) but tractable, so we use it here for the reported statistic.
sys.path.insert(0, '../code')
import my_utils as mu

tt_path  = paths.inp("tt_gfc_50")
pop_path = paths.inp("population")

with rasterio.open(pop_path) as pop_src, rasterio.open(tt_path) as tt_src:
    pop, tt = mu.align_rasters(pop_src, tt_src)

tt = tt.astype(np.float32)
pop = pop.astype(np.float32)

# Filter: valid travel time, positive population
valid = (~np.isnan(tt)) & (pop > 0)
tt_v  = tt[valid]
pop_v = pop[valid]
del tt, pop, valid

pop_weighted_mean = np.average(tt_v, weights=pop_v)
pop_weighted_median_approx = np.percentile(
    np.repeat(tt_v.astype(int), np.clip(pop_v.astype(int), 1, None))[:5_000_000], 50
)

print(f"Population-weighted mean TT   : {pop_weighted_mean:.1f} min")
print(f"Total population in analysis  : {pop_v.sum()/1e9:.2f} billion")
print(f"Pixel count                   : {len(tt_v):,}")

### 2. Gini coefficient of forest access

In [ ]:
def gini_weighted(values, weights):
    """Population-weighted Gini coefficient."""
    idx = np.argsort(values)
    v = values[idx]
    w = weights[idx]
    w_norm = w / w.sum()
    lorenz = np.cumsum(v * w_norm) / np.sum(v * w_norm)
    lorenz = np.concatenate([[0], lorenz])
    cum_pop = np.concatenate([[0], np.cumsum(w_norm)])
    area = np.trapz(lorenz, cum_pop)
    return 1 - 2 * area

gini = gini_weighted(tt_v, pop_v)
print(f"Gini coefficient of TT access : {gini:.3f}")
print(f"(0 = perfect equality, 1 = perfect inequality)")

### 3. Cumulative access summary

In [ ]:
df_global = pd.read_csv(paths.der("population_by_macroregion_traveltime.csv"))
df_sum = df_global.groupby('threshold_min')['population'].sum().reset_index()
total_pop = 7_830_089_931
df_sum['pct'] = 100 * df_sum['population'] / total_pop

print("Threshold | Population (B) | Cumulative %")
print("-" * 45)
for _, row in df_sum.iterrows():
    print(f"  {int(row.threshold_min):3d} min | {row.population/1e9:14.2f} | {row.pct:10.1f}%")

### 4. Weighted Spearman & Pearson correlation (GDP per capita vs mean TT)

In [ ]:
reg = pd.read_csv(paths.der("macroregion_pop_gdp_tt_average.csv"))
reg['continent'] = reg['region'].map(REGION_TO_CONTINENT)

gdp = reg['gdp_pc'].values
tt_reg = reg['time'].values
pop_reg = reg['population'].values.astype(float)

# Standard Spearman (17 regions, unweighted)
r_sp, p_sp = stats.spearmanr(gdp, tt_reg)

# Population-weighted Pearson correlation
w = pop_reg / pop_reg.sum()
gdp_mean_w = np.sum(w * gdp)
tt_mean_w  = np.sum(w * tt_reg)
cov_w   = np.sum(w * (gdp - gdp_mean_w) * (tt_reg - tt_mean_w))
std_gdp = np.sqrt(np.sum(w * (gdp - gdp_mean_w)**2))
std_tt  = np.sqrt(np.sum(w * (tt_reg - tt_mean_w)**2))
r_wpearson = cov_w / (std_gdp * std_tt)

# Bootstrap p-value for weighted Pearson
np.random.seed(42)
n_boot = 10_000
r_boot = []
for _ in range(n_boot):
    idx = np.random.choice(len(gdp), size=len(gdp), replace=True, p=w)
    gboot, tboot = gdp[idx], tt_reg[idx]
    wboot = np.ones(len(idx)) / len(idx)
    gm = gboot.mean(); tm = tboot.mean()
    c = np.mean((gboot-gm)*(tboot-tm))
    sg = np.std(gboot); st = np.std(tboot)
    r_boot.append(c / (sg * st) if sg * st > 0 else 0)
p_wpearson = np.mean(np.abs(r_boot) >= np.abs(r_wpearson))

print(f"Spearman r (unweighted, n=17) : {r_sp:.3f}  (p={p_sp:.3f})")
print(f"Pearson r  (pop-weighted)     : {r_wpearson:.3f}  (bootstrap p={p_wpearson:.3f})")
print()
print("Interpretation:")
neg = "negative" if r_wpearson < 0 else "positive"
print(f"  Weighted r = {r_wpearson:.3f} ({neg} correlation)")
print("  Wealthier regions tend to have", "shorter" if r_wpearson < 0 else "longer", "travel times to forest")

### 5. Regional summary table

In [ ]:
reg_sorted = reg.sort_values('time')
reg_sorted['pop_M'] = (reg_sorted['population'] / 1e6).round(1)
reg_sorted['gdp_pc'] = reg_sorted['gdp_pc'].round(0).astype(int)
reg_sorted['time'] = reg_sorted['time'].round(1)
print(reg_sorted[['region', 'continent', 'time', 'gdp_pc', 'pop_M']].to_string(index=False))